# E4-B — Arsitektur, modif B: padding=1 + BatchNorm
**Interpretasi dari E4-A:** mengganti sigmoid+BCE → logits+CE langsung menaikkan test acc dari 10% → **43.7%**. Tapi kurva masih menanjak lambat (train acc baru 37.8% di epoch 3, loss masih turun) → model *underfitting*, training belum konvergen. Dugaan: tanpa normalisasi antar-layer, distribusi aktivasi bergeser terus (internal covariate shift) sehingga lr kecil 0.0025 lambat merambat.

**Modif B:** pada arsitektur yang sama, tambahkan:
- `padding=1` di semua conv → spatial size terjaga (32→16→8, bukan menyusut 32→30→28→…), tepi gambar ikut terbaca
- `BatchNorm2d` setelah tiap conv → aktivasi distabilkan, gradient mengalir lebih baik, training lebih cepat konvergen

Hyperparameter tetap SGD lr=0.0025, batch 10 — isolasi murni efek arsitektur.

## Run

In [1]:
from exp_common import *
model = ConvNetB()
res = run_experiment(
    'E4_arch_B', model,
    train_files=['data_batch_1', 'data_batch_3', 'data_batch_4', 'data_batch_5'],
    val_files=['data_batch_2'],
    epochs=3, batch_size=10, lr=0.0025, optimizer_type='sgd',
    loss_mode='ce', augment=True, fix_channels=True)


===== E4_arch_B =====
device=cuda, epochs=3, batch=10, lr=0.0025, opt=sgd, loss=ce, augment=True, fix_channels=True
params total=41,034 trainable=41,034
train=40000 val=10000


epoch 0: train_loss=1.7311 train_acc=0.3416 val_loss=1.5110 val_acc=0.4448 (21.6s, 1850 samples/s)


epoch 1: train_loss=1.4373 train_acc=0.4701 val_loss=1.3278 val_acc=0.5126 (20.8s, 1918 samples/s)


epoch 2: train_loss=1.3167 train_acc=0.5213 val_loss=1.2165 val_acc=0.5695 (24.8s, 1615 samples/s)
peak GPU mem: 57 MB


TEST ACC = 0.5673  (eval 0.7s, train 67.2s)
saved: C:\Users\dgash\Downloads\ml_task_cnn\experiments\results\E4_arch_B.json
